# Pendulum swing-up — value iteration vs LQR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/teaching/courses/gro860/labs/pendulum_swing_up_vi_vs_lqr.ipynb)

Two controllers for the same swing-up problem and quadratic cost:

1. **Value iteration (VI)**: global non-linear policy from dynamic programming.
2. **LQR**: local linear feedback from linearized dynamics.

This page uses [minilink](https://github.com/alx87grd/minilink).


In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minilink.core.diagram import DiagramSystem
from minilink.core.trajectory import Trajectory
from minilink.dynamics.catalog.pendulum.pendulum import Pendulum
from minilink.planning.policy_synthesis import plotting
from minilink.planning.policy_synthesis.discretizer import StateSpaceGrid
from minilink.planning.policy_synthesis.dp import (
    DynamicProgrammingOptions,
    DynamicProgrammingPlanner,
)
from minilink.planning.problems import PlanningProblem


## 1. Plant

Here we load the pendulum model and set the domain for the state $x = [\theta, \dot\theta]$ and the torque $u$. The hanging-down position is $\theta = 0$; the upright target is $\bar x = [-\pi,\; 0]$.


In [ ]:
UPRIGHT = np.array([-np.pi, 0.0])  # target (upright) and LQR linearization point
X0 = np.array([0.0, 0.0])  # hanging down
TORQUE = 5.0
DT = 0.05
TF = 10.0
X_GRID = (201, 201)
U_GRID = (21,)
TOL = 0.1
INF = 500.0
Q = np.diag([1.0, 1.0])
R = np.diag([1.0])


def make_pendulum():
    plant = Pendulum()
    plant.state.lower_bound = np.array([-2.0 * np.pi, -2.0 * np.pi])
    plant.state.upper_bound = np.array([+2.0 * np.pi, +2.0 * np.pi])
    plant.inputs["u"].lower_bound = np.array([-TORQUE])
    plant.inputs["u"].upper_bound = np.array([+TORQUE])
    plant.x0 = X0.copy()
    return plant


plant = make_pendulum()


## 2. Cost function

Default quadratic cost:

$$J = \int_0^{t_f} \big( (x-\bar x)' Q (x-\bar x) + u' R u \big)\, dt$$

with $Q = I$ and $R = I$.


In [ ]:
from minilink.core.costs import QuadraticCost

cost = QuadraticCost.from_system(plant, xbar=UPRIGHT, Q=Q, R=R)

print("Target:", UPRIGHT)
print("Q=\n", cost.Q)
print("R=\n", cost.R)


## 3. Planning problem

The optimal-control problem is: drive the pendulum to $\bar x$, while minimizing the cost above, subject to the torque limits.


In [ ]:
problem = PlanningProblem(plant, x_goal=UPRIGHT, cost=cost)


## 4. Value iteration

Discretize the state and torque, then solve the Bellman equation. The 2-D state is a $201\times 201$ grid, the torque 21 levels, the time step $0.05\,\mathrm{s}$.


In [ ]:
grid = StateSpaceGrid(problem, x_grid_shape=X_GRID, u_grid_shape=U_GRID, dt=DT)

planner = DynamicProgrammingPlanner(
    problem,
    grid=grid,
    options=DynamicProgrammingOptions(
        alpha=1.0,
        tol=TOL,
        max_iterations=2000,
        out_of_bound_cost=INF,
        verbose=True,
    ),
)

result = planner.solve().policy
result = planner.clean_infeasible_set()
vi_ctl = result.controller()


## 5. LQR

Linearize at $\bar x$ and synthesize $u = \bar u - K(x-\bar x)$. Same $Q$, $R$, and plant as value iteration.


In [ ]:
from minilink.control.lqr import lqr_at_operating_point

lqr_ctl = lqr_at_operating_point(make_pendulum(), UPRIGHT, Q, R)
K = lqr_ctl.params["K"]
print("LQR gain K =", np.round(K, 3))


## 6. Control laws

LQR is a linear map; VI is a non-linear map that follows the natural dynamics. Near $\bar x$ they look similar; globally LQR asks for much larger torques.


In [ ]:
K_row = lqr_ctl.params["K"][0]
ubar = lqr_ctl.params["ubar"][0]
lqr_law = ubar - (grid.states - UPRIGHT) @ K_row

plotting.plot_policy(grid, result.pi)
plotting.plot_value(
    grid, lqr_law, vmin=-TORQUE, vmax=TORQUE, cmap="bwr", title="LQR control law"
)


## 7. Closed-loop simulation

Both from the hanging position $[\theta=0,\;\dot\theta=0]$. LQR goes straight to the goal with large torque; VI pumps, then swings up.


In [ ]:
def closed_loop(controller, x0, name):
    """Wire a state-feedback controller to a fresh copy of the pendulum."""
    plant = make_pendulum()
    plant.x0 = np.array(x0)
    diagram = DiagramSystem()
    diagram.add_subsystem(controller, "ctl")
    diagram.add_subsystem(plant, "plant")
    diagram.connect("plant", "y", "ctl", "x")
    diagram.connect("ctl", "u", "plant", "u")
    diagram.name = name
    diagram.camera_scale = 2.0
    n_steps = int(TF / DT) + 1  # same step as the DP discretization
    traj = diagram.compute_trajectory(tf=TF, n_steps=n_steps, solver="euler")
    return diagram, plant, traj


def applied_u(controller, traj):
    """Reconstruct u(t) from a controller that implements action(x)."""
    return np.array([controller.action(x) for x in traj.x.T]).T

cl_vi, plant_vi, traj_vi = closed_loop(vi_ctl, X0, "Pendulum with VI")
cl_lqr, plant_lqr, traj_lqr = closed_loop(lqr_ctl, X0, "Pendulum with LQR")

cl_vi.plot_trajectory(traj_vi)
cl_lqr.plot_trajectory(traj_lqr)


## 8. Animation — VI


In [ ]:
cl_vi.animate(traj_vi)


## 8. Animation — LQR


In [ ]:
cl_lqr.animate(traj_lqr)


## 9. Phase plane

VI rides the natural dynamics instead of fighting them.


In [ ]:
plant_vi.plot_phase_plane(traj_vi)
plant_lqr.plot_phase_plane(traj_lqr)


## 10. Performance


In [ ]:
u_lqr = np.clip(
    ubar - (traj_lqr.x.T - UPRIGHT) @ K_row, -TORQUE, TORQUE
).reshape(1, -1)
traj_vi_cost = cost.evaluate_trajectory(
    Trajectory(t=traj_vi.t, x=traj_vi.x, u=applied_u(vi_ctl, traj_vi))
)
traj_lqr_cost = cost.evaluate_trajectory(
    Trajectory(t=traj_lqr.t, x=traj_lqr.x, u=u_lqr)
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(traj_vi_cost.t, traj_vi_cost.signals["cost"][0], label="VI")
ax.plot(traj_lqr_cost.t, traj_lqr_cost.signals["cost"][0], label="LQR")
ax.set_xlabel("t [s]")
ax.set_ylabel("$J$")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

